In [26]:
import os
import pandas as pd
import numpy as np
import torch
import gymnasium as gym
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. 安全加载函数 (适配较新版本 PyTorch 的加载要求)
#    不再覆盖全局 torch.load，避免多次执行单元导致递归问题
# ---------------------------------------------------------
import torch.serialization as _torch_serialization
torch.load = _torch_serialization.load  # 强制恢复官方实现，清理历史补丁副作用


def safe_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _torch_serialization.load(*args, **kwargs)

def _load_actor_weights_compat(model, checkpoint_path, device):
    """兼容加载三种常见保存格式：state_dict / nn.Module / dict checkpoint。"""
    obj = safe_torch_load(checkpoint_path, map_location=device)

    # 1) 直接是 state_dict
    if isinstance(obj, dict) and all(isinstance(k, str) for k in obj.keys()):
        # 有些 checkpoint 会包一层，如 {'act': state_dict} 或 {'actor': nn.Module}
        if 'state_dict' in obj and isinstance(obj['state_dict'], dict):
            model.load_state_dict(obj['state_dict'], strict=False)
            return "state_dict"

        for key in ('act', 'actor', 'model', 'net'):
            if key in obj:
                inner = obj[key]
                if isinstance(inner, dict):
                    model.load_state_dict(inner, strict=False)
                    return f"dict['{key}']"
                if isinstance(inner, torch.nn.Module):
                    model.load_state_dict(inner.state_dict(), strict=False)
                    return f"module['{key}']"

        # 若键看起来像参数名，按普通 state_dict 处理
        if any('.weight' in k or '.bias' in k for k in obj.keys()):
            model.load_state_dict(obj, strict=False)
            return "raw_state_dict"

    # 2) 直接保存了 nn.Module（你当前报错正是这种情况）
    if isinstance(obj, torch.nn.Module):
        model.load_state_dict(obj.state_dict(), strict=False)
        return "nn.Module"

    raise TypeError(f"Unsupported checkpoint type: {type(obj)} from {checkpoint_path}")


# ---------------------------------------------------------
# 2. 导入 ElegantRL 组件
# ---------------------------------------------------------
from elegantrl.agents import AgentDQN
from elegantrl.train.run import train_agent
try:
    from elegantrl.train.config import Config as Arguments
except ImportError:
    from elegantrl.train.config import Arguments

from env_wrapper.dqn_env_wrapper import FastDiscreteCryptoEnv


In [27]:
# ---------------------------------------------------------
# 4. 数据处理逻辑
# ---------------------------------------------------------
def load_local_crypto_data(csv_path="./data/BTC_1sec.csv", npy_path="./data/BTC_1sec_predict.npy"):
    print(f"[INFO] 正在加载高频数据...")
    if not os.path.exists(csv_path) or not os.path.exists(npy_path):
        print("[WARNING] 文件丢失，生成 MOCK 数据用于测试。请确保 /data 目录下存在所需文件。")
        raise FileNotFoundError(f"缺少数据文件：{csv_path} 或 {npy_path}。请确保它们存在于 /data 目录下。")
        # np.random.seed(42)
        # mock_len = 5000
        # raw_df = pd.DataFrame({'SystemTime': pd.date_range('2021-04-07', periods=mock_len, freq='S'),
        #                        'MidPrice': np.cumsum(np.random.randn(mock_len)) + 50000})
        # features = np.random.randn(mock_len, 8) 
    else:
        raw_df = pd.read_csv(csv_path)
        features = np.load(npy_path)

    # 对齐长度（如果有 RNN Lookback Window）
    diff = len(raw_df) - len(features)
    if diff > 0:
        raw_df = raw_df.iloc[diff:].reset_index(drop=True)

    finrl_df = pd.DataFrame()

    price_col = 'midpoint' if 'midpoint' in raw_df.columns else raw_df.columns[1]
    finrl_df['close'] = pd.to_numeric(raw_df[price_col], errors='coerce').astype(float)
    
    feature_cols = [f"rnn_feature_{i}" for i in range(features.shape[1])]
    for i, col in enumerate(feature_cols):
        finrl_df[col] = features[:, i].astype(float)

    # 填充缺失值，防止脏数据影响特征输入
    finrl_df = finrl_df.fillna(method='ffill').fillna(0.0)
    return finrl_df, feature_cols

def split_train_test_physical(df):
    test_size = 2700  # 论文标准：测试集为最后 45 分钟的数据
    df_train = df.iloc[:-test_size].copy()
    df_test = df.iloc[-test_size:].copy()
    return df_train, df_test

In [28]:
df, PAPER_INDICATORS = load_local_crypto_data()
df_train, df_test = split_train_test_physical(df) # length: 23000 train, 2700 test

print(f"[数据加载完成]")
print(f"  训练集: {len(df_train)} 条数据")
print(f"  测试集: {len(df_test)} 条数据")
print(f"  特征数量: {len(PAPER_INDICATORS)}")

# 2. 计算出框架需要的必备参数
state_dim = 3 + len(PAPER_INDICATORS) # 3 个基本状态 + 论文特征数量
action_dim = 3 
max_step = len(df_train) - 1

[INFO] 正在加载高频数据...
[数据加载完成]
  训练集: 820982 条数据
  测试集: 2700 条数据
  特征数量: 8


In [29]:
print(df_train.head())
# 打印raw_df中的columns name
# print(raw_df.columns.tolist())

       close  rnn_feature_0  rnn_feature_1  rnn_feature_2  rnn_feature_3  \
0  56100.005      -0.097931      -0.031718       0.012926       0.019522   
1  56099.375      -0.134023      -0.067983      -0.025429      -0.017401   
2  56100.005      -0.165179      -0.098757      -0.054769      -0.037389   
3  56100.005      -0.175863      -0.119987      -0.079141      -0.053333   
4  56100.015      -0.173713      -0.135205      -0.101312      -0.066894   

   rnn_feature_4  rnn_feature_5  rnn_feature_6  rnn_feature_7  
0       0.014166       0.018815       0.036858       0.048783  
1      -0.015060      -0.005570       0.014706       0.029882  
2      -0.023390      -0.010219       0.008212       0.020741  
3      -0.028855      -0.013482       0.000399       0.008215  
4      -0.031418      -0.014660      -0.007651      -0.006596  


In [ ]:
# 5. DQN 训练与评估配置
# ---------------------------------------------------------
def setup_dqn_args(env_params, cwd_path):
    args = Arguments(agent_class=AgentDQN, env_class=FastDiscreteCryptoEnv)
    args.env_args = env_params
    args.env_name = env_params['env_name']

    # [cite_start]论文设定：三个 128 单元的前馈神经网络隐藏层 [cite: 771]
    args.net_dims = (128, 128, 128)
    
    args.state_dim = env_params['state_dim']
    args.action_dim = env_params['action_dim']
    args.if_discrete = True

    # 提高学习率，让模型能够更快学习到有效策略
    args.learning_rate = 2e-6
    args.batch_size = 512

    # 增加探索率，确保训练时有足够的探索
    args.explore_rate = 0.005
    
    args.eval_gap = 500
    args.save_gap = 200

    args.target_step = 1000 # 每 2000 步更新一次目标网络
    args.break_step = 100000  # total 200k

    args.worker_num = 1
    args.eval_proc_num = 0    # 避免 Windows 平台下的评估器多进程闪退
    args.cwd = cwd_path
    args.if_remove = False
    args.if_save = True

    return args

In [31]:
def real_test_inference(test_df, indicators, args, env_params):
    # 构建纯 NumPy 推理环境
    env = FastDiscreteCryptoEnv(
        df=test_df,
        indicators=indicators,
        cost_pct=env_params['cost_pct']
    )

    agent = AgentDQN(args.net_dims, args.state_dim, args.action_dim)
    agent.save_or_load_agent(args.cwd, if_save=False)

    agent.act.eval()
    agent.explore_rate = 0.0  # 推理时禁用 epsilon-greedy

    res = env.reset()
    state = res[0] if isinstance(res, tuple) else res
    done = False

    # 统计动作分布
    action_counts = {0: 0, 1: 0, 2: 0}  # 0:卖出, 1:持有, 2:买入
    print("[INFO] 正在执行 45 分钟逐秒回测推理...")
    while not done:
        s_tensor = torch.as_tensor((state,), dtype=torch.float32, device=agent.device)
        with torch.no_grad():
            q_values = agent.act(s_tensor)
            action = q_values.item()
            # action = int(torch.argmax(q_values, dim=-1).reshape(-1)[0].item())

        action_counts[action] += 1

        # 环境 step 期望离散 int 动作
        step_res = env.step(action)
        state, reward, done, trunc, _ = step_res if len(step_res) == 5 else (*step_res[:3], False, {})
        done = done or trunc

    # 打印动作统计
    total_actions = sum(action_counts.values())
    print(f"\n[动作统计]")
    print(f"  卖出(0): {action_counts[0]} 次 ({action_counts[0] / total_actions * 100:.1f}%)")
    print(f"  持有(1): {action_counts[1]} 次 ({action_counts[1] / total_actions * 100:.1f}%)")
    print(f"  买入(2): {action_counts[2]} 次 ({action_counts[2] / total_actions * 100:.1f}%)")

    # 导出测试结果
    result_df = pd.DataFrame({'account_value': env.asset_memory})
    return result_df

In [ ]:
# 3. 补齐 ElegantRL 多进程装配环境所必需的参数
env_params = {
    "env_name": "FastCryptoEnv",
    "df": df_train,
    "indicators": PAPER_INDICATORS,
    "cost_pct": 0.0002,  # 0.0002 - 论文设定的交易成本
    "state_dim": state_dim,
    "action_dim": action_dim,
    "if_discrete": True,
    "max_step": max_step,
    "target_return": 10.0,
    "initial_amount": 1000000,
}

print(f"\n[STEP 1] 开始 DQN 训练（续训模式）...")
print(f"  学习率: 1e-4")
print(f"  交易成本: {env_params['cost_pct']*100:.03f}%")

cwd_path = "./checkpoints/dqn_crypto_hft_fast"
os.makedirs(cwd_path, exist_ok=True)
args = setup_dqn_args(env_params, cwd_path)

try:
    # 直接续训（会从 cwd 中已有参数与记录继续）
    train_agent(args)
    print("\n[✓] 训练完成！")
except Exception as e:
    import traceback
    print(f"\n[WARNING] 训练阶段遇到异常，堆栈信息如下:")
    traceback.print_exc()

print(f"\n[STEP 2] 准备生成回测结果...")

try:
    # 用测试集评估（论文设定：最后 45 分钟）
    test_results = real_test_inference(df_test, PAPER_INDICATORS, args, env_params)

    output_file = "dqn_crypto_hft_results.csv"
    test_results.to_csv(output_file, index=False)
    print(f"\n[✓] 成功！结果已保存至 '{output_file}'。")
    print(f"======================================")
    print(f"初始资产: {test_results['account_value'].iloc[0]:.2f}")
    print(f"最终资产: {test_results['account_value'].iloc[-1]:.2f}")
    print(f"总收益率: {(test_results['account_value'].iloc[-1] / test_results['account_value'].iloc[0] - 1) * 100:.2f}%")
    print(f"======================================")
    print(f"\n最终资金曲线前5条预览:\n{test_results.head()}")
    print(f"最终资金曲线最后5条预览:\n{test_results.tail()}")
except Exception as e:
    import traceback
    print(f"\n[ERROR] 回测推理阶段崩溃:")
    traceback.print_exc()



[STEP 1] 开始 DQN 训练（续训模式）...
  学习率: 1e-4
  交易成本: 0.020%
| train_agent_multiprocessing() with GPU_ID 0
| Arguments Keep cwd: ./checkpoints/dqn_crypto_hft_fast
[✓] ElegantRL 导入完成 - 使用 AgentDQN
[✓] ElegantRL 导入完成 - 使用 AgentDQN
[✓] ElegantRL 导入完成 - 使用 AgentDQN
[✓] ElegantRL 导入完成 - 使用 AgentDQN
| Evaluator:
| `step`: Number of samples, or total training steps, or running times of `env.step()`.
| `time`: Time spent from the start of training to this moment.
| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode.
| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode.
| `avgS`: Average of steps in an episode.
| `objC`: Objective of Critic network. Or call it loss function of critic network.
| `objA`: Objective of Actor network. It is the average Q value of the critic network.
################################################################################
ID     Step    Time |    avgR   stdR   avgS  stdS |    expR   objC   objA